In [ ]:
# [Cell 1] Imports
import os
import time
import random
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[System] Device: {device}")
if torch.cuda.is_available():
    print(f"[System] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[System] Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")



In [ ]:
# [Cell 2] Configuration
class Config:
    dataroot = "./monet_jpg"
    image_size = 128
    batch_size = 32 if torch.cuda.is_available() else 8
    workers = 0
    
    nz = 100
    ngf = 64
    ndf = 64
    nc = 3
    num_epochs = 100
    lr = 0.0002
    beta1 = 0.5
    beta2 = 0.999
    sample_interval = 20  # Save images every 20 epochs
    sample_size = 64

config = Config()
print(f"[Config] Dataset: {os.path.abspath(config.dataroot)}")
print(f"[Config] Image size: {config.image_size}")
print(f"[Config] Batch size: {config.batch_size}")
print(f"[Config] Workers: {config.workers}")
print(f"[Config] Epochs: {config.num_epochs}")



In [ ]:
# [Cell 3] Data Loading
print("[Data] Loading dataset...")

class MonetDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        self.image_paths = []
        for ext in exts:
            self.image_paths.extend(glob.glob(os.path.join(root_dir, ext)))
        self.image_paths = list(set(self.image_paths))
        
        if len(self.image_paths) == 0:
            raise FileNotFoundError(f"No images found in {root_dir}")
        print(f"[Data] Found {len(self.image_paths)} images")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, 0

transform = transforms.Compose([
    transforms.Resize(config.image_size),
    transforms.CenterCrop(config.image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = MonetDataset(root_dir=config.dataroot, transform=transform)
dataloader = DataLoader(
    dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.workers,
    pin_memory=False,
    drop_last=True
)
print(f"[Data] Batches: {len(dataloader)}")

# Show real samples
def show_real_images():
    real_batch = next(iter(dataloader))
    plt.figure(figsize=(10, 8))
    plt.axis("off")
    plt.title("Real Monet Paintings")
    grid = vutils.make_grid(real_batch[0][:64], padding=2, normalize=True)
    plt.imshow(np.transpose(grid.cpu(), (1, 2, 0)))
    plt.show()

show_real_images()



In [ ]:
# [Cell 4] Generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(config.nz, config.ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(config.ngf * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(config.ngf * 16, config.ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(config.ngf * 8, config.ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(config.ngf * 4, config.ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(config.ngf * 2, config.ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(config.ngf, config.nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

netG = Generator().to(device)
print(f"[Model] Generator params: {sum(p.numel() for p in netG.parameters()):,}")



In [ ]:
# [Cell 5] Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(config.nc, config.ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(config.ndf, config.ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(config.ndf * 2, config.ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(config.ndf * 4, config.ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(config.ndf * 8, config.ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config.ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(config.ndf * 16, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)

netD = Discriminator().to(device)
print(f"[Model] Discriminator params: {sum(p.numel() for p in netD.parameters()):,}")

